In [1]:
import utils
import pandas as pd
import os
from collections import defaultdict
from pathlib import Path

In [2]:
# load the table with all human targets and the corresponding drugs
gene_table = pd.read_excel("../data/gene_drugs.xlsx")
gene_t = (
    gene_table
    .assign(drug=gene_table.iloc[:, 1].str.split(";"))
    .explode("drug")
    .groupby(gene_table.columns[0])["drug"]
    .unique()
    .apply(list)
    .to_dict()
)

# make a dictionary on drugs
dict_drugs = defaultdict(set)
for gene, drugs in gene_t.items():
    for drug in drugs:
        dict_drugs[drug].add(gene)

# load the the table with drugs that have ATC code "L" and find the corresponding targets
db_table = pd.read_excel("../data/atc_l_drugs.xlsx")

genes_df = (
    pd.Series(dict_drugs, name="Genes")
    .rename_axis("ID")
    .reset_index()
)

db_table = db_table.merge(genes_df, on="ID", how="left")

db_table["Genes"] = (db_table["Genes"]
    .astype("string")
    .str[1:-1]
    .str.replace(r"[\'\s]+", "", regex=True)
                    )
print("The number of drugs retrieved (with and without targets):",len(db_table))    

db_table = (db_table
    .dropna(subset=["Genes"])
    .reset_index(drop=True))

print("The number of drugs with targets:", len(db_table))

db_table.to_excel("../data/drugs_with_targets.xlsx")

# make a list with all unique targets and save them

all_targets = (
    db_table.iloc[:, 2]
    .dropna()
    .str.split(",")
    .explode()
    .unique()
    .tolist()
)

print("The number of drug targets:", len(all_targets))

with open('../data/DrugTargets.txt', 'w') as f:
    for gene in all_targets:
        f.write(f"{gene}\n")


The number of drugs retrieved (with and without targets): 651
The number of drugs with targets: 422
The number of drug targets: 458


In [3]:
# read the file with glioblastoma dependencies
with open("../data/GBM_dependencies.txt", "r") as f:
    ess_list = [line.strip() for line in f]
    
# the mutations for each case is in utils.mutation_t
print("The number of cases with without mutations:", 150-len(utils.mutation_t))

# the list of cases = utils.patient_ids

The number of cases with without mutations: 8


In [4]:
#compute setA, setB and setMDown and save them

setA = {}
setB = {}
setMDown = {}

for patient in utils.patient_ids:
    base_up_down = utils.up_dict.get(patient, []) + utils.down_dict.get(patient, [])
    mutations = utils.mutation_t.get(patient, [])
    setA[patient] = list(set(base_up_down + mutations + all_targets))
    setB[patient] = list(set(base_up_down + mutations + ess_list))

for patient in utils.patient_ids:
    mutations = utils.mutation_t.get(patient, [])
    setMDown[patient] = list(set(utils.down_dict.get(patient, []) + mutations))

def save_set(path, set_to_save):
    path_set = Path(path)
    path_set.mkdir(exist_ok=True) 
    for patient, genes in set_to_save.items():
        file_path = Path.cwd()/path_set/ f"{patient}.txt"
        with file_path.open("w") as f:
            for gene in genes:
                f.write(f"{gene}\n")

save_set("../data/SetMDown", setMDown)
save_set("../data/SetA", setA)
save_set("../data/SetB", setB)